# Baghewala Field Digital Twin — SRP Telemetry Anomaly Detection
**Objective:** Train an Unsupervised Isolation Forest on 46,112 hourly multi-variate telemetry snapshots from `data/external/srp_telemetry_features_master.parquet`.

Detects:
- Severe fluid pound (collapsed fillage & dynacard work loop)
- Rod floating / slack conditions ($MPRL 	o 0$)
- Gas locking & excessive cyclic load range ($\Delta W$)

Artifacts saved directly to `models/anomaly/`.


In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

MODELS_DIR = Path("models")
telem_path = Path("data/external/srp_telemetry_features_master.parquet")
df_telem = pd.read_parquet(telem_path)
print(f"Loaded {len(df_telem)} pivoted telemetry records across {df_telem['well_id'].nunique()} wells.")


### 1. Telemetry Feature Vector Preparation

In [ ]:
# Map standardized columns to feature vector
df_telem['rod_load_range'] = df_telem['pprl_kg'] - df_telem['mprl_kg']
df_telem['slack_ratio'] = df_telem['mprl_kg'] / (df_telem['pprl_kg'] + 1.0)

feature_cols = [
    "spm", "pprl_kg", "mprl_kg", "dynamometer_area_work",
    "pump_fillage_pct", "tubing_pressure_bar", "casing_pressure_bar",
    "rod_load_range", "slack_ratio"
]

X_telem = df_telem[feature_cols].ffill().bfill().fillna(0).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_telem)
print(f"Feature matrix shape: {X_scaled.shape}")


### 2. Train Unsupervised Isolation Forest Model

In [ ]:
iso_forest = IsolationForest(
    n_estimators=120,
    contamination=0.03,  # Expect 3% operational anomalies
    random_state=42
)
iso_forest.fit(X_scaled)

preds = iso_forest.predict(X_scaled)
anom_count = int(np.sum(preds == -1))
print(f"Anomaly Detection Results:")
print(f"  Total Snapshots Evaluated: {len(X_scaled)}")
print(f"  Normal Operating Points:    {int(np.sum(preds == 1))}")
print(f"  Flagged Anomalies:         {anom_count} ({anom_count / len(X_scaled) * 100:.2f}%)")

anom_dir = MODELS_DIR / "anomaly"
anom_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(iso_forest, anom_dir / "rod_telemetry_isolation_forest.joblib")
joblib.dump(scaler, anom_dir / "telemetry_scaler.joblib")
joblib.dump(feature_cols, anom_dir / "telemetry_feature_names.joblib")
print(f"Saved artifacts to {anom_dir}")
